# Distil the recitation model for phones

Runs the whole distillation on a free Colab GPU (about 1–2 hours):

1. The large Quran model (teacher) labels 8 reciters' audio, frame by frame.
2. Two phone-sized students learn to copy it: 12 layers and 8 layers (v2: speed perturbation
   and extra weight on letters).
3. Both are tested on Minshawi's al-Fatiha and al-Mulk against the teacher.
4. You download `students-v2.zip` and attach it in the Claude session.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
Keep this tab open until the last cell offers the download.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Get the spike code (only the tools/recitation-spike folder of the fork)
!rm -rf quran_android
!git clone -q --depth 1 --branch feature/recitation-tracking-spike --filter=blob:none --sparse https://github.com/nkaoud-app/quran_android
!cd quran_android && git sparse-checkout set tools/recitation-spike
%cd /content/quran_android/tools/recitation-spike
!pip install -q transformers soundfile
%env SPIKE_DATA=/content/data

## 1. Teacher labels the training audio (~20–40 min, mostly downloading)

In [ ]:
!python distill.py label 2>&1 | grep --line-buffered -v -i "warn" 

## 2. Train the students (~15–30 min each)

In [ ]:
!python distill.py train --out /content/students/s12 --layers 12 --steps 6000 --batch 16 --lr 1e-4 --log-every 300 2>&1 | grep --line-buffered -E "^step|recordings|Error" 

In [ ]:
!python distill.py train --out /content/students/s8 --layers 8 --steps 6000 --batch 16 --lr 1e-4 --log-every 300 2>&1 | grep --line-buffered -E "^step|recordings|Error" 

## 3. Check the students on held-out recitations

In [ ]:
for model in ["/content/students/s12", "/content/students/s8"]:
    !python evaluate.py --suras 1 67 --model {model} 2>&1 | grep -E "^model|^sura|Error" 

## 4. Download the students

Saved in half precision to keep the file small (~300 MB). Attach `students-v2.zip` in the Claude session.

In [ ]:
import shutil, torch
from transformers import Wav2Vec2ForCTC
for name in ["s12", "s8"]:
    model = Wav2Vec2ForCTC.from_pretrained(f"/content/students/{name}").half()
    model.save_pretrained(f"/content/export/{name}")
    shutil.copy(f"/content/students/{name}/classes.json", f"/content/export/{name}/")
shutil.make_archive("/content/students-v2", "zip", "/content/export")
from google.colab import files
files.download("/content/students-v2.zip")